In [ ]:
# Cell 1: Load GitHub PAT from Kaggle Secrets
from kaggle_secrets import UserSecretsClient
import os

secrets = UserSecretsClient()
pat = secrets.get_secret('GITHUB_PAT')
os.environ['GITHUB_PAT'] = pat
print('PAT loaded OK')

In [ ]:
# Cell 2: Clone repo (branch dengan fixes)
import os
pat = os.environ['GITHUB_PAT']

# cd to safe dir first to avoid getcwd error when rm -rf deletes cwd
%cd /kaggle/working
!rm -rf EMA-SKD
!git clone https://{pat}@github.com/almaas-izdihar/ema-skd EMA-SKD
%cd EMA-SKD
!git checkout experiment/ablation-baseline
!git log --oneline -5

In [ ]:
# Cell 3: Verify GPU
!nvidia-smi

In [ ]:
# Cell 4: Run 1 — Baseline (L_CE only, no EHSKD)
# Target paper: 75.55 ± 0.09
!python main.py \
  --data_type cifar100 \
  --data_path /kaggle/working/data \
  --classifier_type ResNet18 \
  --batch_size 128 \
  --end_epoch 200 \
  --workers 2 \
  --seed 2024 \
  --experiment_type run1_baseline

In [ ]:
# Cell 5: Run 2 — Full EMA-SKD (Fix A+B: α added to L_KD2 and L_Refine)
# Fix A: mixup_loss * args.weight (was × 1.0, now × 4.0)
# Fix B: weight2 default 1.0 → 4.0 (L_Refine now × 4.0)
# Target paper: 79.19 ± 0.15
!python main.py \
  --data_type cifar100 \
  --data_path /kaggle/working/data \
  --classifier_type ResNet18 \
  --batch_size 128 \
  --end_epoch 200 \
  --workers 2 \
  --seed 2024 \
  --beta 0.5 \
  --EHSKD \
  --experiment_type run2_emaskd_fixed_alpha

In [ ]:
# Cell 6: Evaluation & Visualization
import glob, re
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

def parse_log(path):
    rows = []
    with open(path) as f:
        for line in f:
            if '[val]' not in line:
                continue
            def g(key):
                m = re.search(rf'\[{key} ([^\]]+)\]', line)
                return float(m.group(1)) if m else None
            rows.append({
                'epoch':      int(re.search(r'\[Epoch (\d+)\]', line).group(1)),
                'val_loss':   g('val_loss'),
                'top1':       g('val_top1_acc'),
                'top5':       g('val_top5_acc'),
                'ece':        g('ECE'),
                'aurc':       g('AURC'),
                'eaurc':      g('EAURC'),
            })
    return pd.DataFrame(rows).set_index('epoch')

def find_log(pattern):
    matches = sorted(glob.glob(f'models/{pattern}/log/log.txt'))
    if not matches:
        raise FileNotFoundError(f'No log found for pattern: {pattern}')
    return matches[-1]

baseline_log = find_log('*run1_baseline*')
fullskd_log  = find_log('*run2_emaskd_fixed_alpha*')

df_base = parse_log(baseline_log)
df_full = parse_log(fullskd_log)

print('Baseline log :', baseline_log)
print('EMA-SKD fixed:', fullskd_log)
print()

last_base = df_base.iloc[-1]
last_full = df_full.iloc[-1]

summary = pd.DataFrame({
    'Metric':    ['Top-1 Acc (%)', 'Top-5 Acc (%)', 'ECE (↓)', 'AURC×10³ (↓)', 'EAURC×10³ (↓)'],
    'Baseline':  [last_base.top1, last_base.top5, last_base.ece, last_base.aurc, last_base.eaurc],
    'EMA-SKD':   [last_full.top1, last_full.top5, last_full.ece, last_full.aurc, last_full.eaurc],
})
summary['Δ'] = summary['EMA-SKD'] - summary['Baseline']
print(summary.to_string(index=False, float_format=lambda x: f'{x:.3f}'))
print()
print(f'Paper targets  — Baseline: 75.55 ± 0.09  |  EMA-SKD: 79.19 ± 0.15')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('EMA-SKD (Fix A+B) vs Baseline — CIFAR-100 / ResNet18', fontsize=13)

for ax, (col, title) in zip(axes, [('top1','Top-1 Accuracy (%)'), ('val_loss','Val Loss'), ('ece','ECE (↓)')]):
    ax.plot(df_base.index, df_base[col], label='Baseline', marker='o', linewidth=1.5)
    ax.plot(df_full.index, df_full[col], label='EMA-SKD (fixed)', marker='s', linewidth=1.5)
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('eval_curves_fixed.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: eval_curves_fixed.png')